# 04 · ¿Qué decisión mejora ForestGuard?
**CRISP-ML(Q) · Comprensión del problema y de los datos.** Una CAR dispone de 20 inspecciones por ciclo. Cada observación representa una celda de unos 1 km al cierre del año T. Se puntúa la probabilidad de perder al menos 2 ha de cobertura arbórea en T+1. La etiqueta Hansen incluye incendios y plantaciones. Una alerta requiere verificación de campo.

**PEAS.** Desempeño: precisión entre las 20 visitas, AP y recall con control de falsas alarmas. Entorno: zonas forestales de la tesela colombiana 10N_080W. Actuadores: lista priorizada, mapa y reporte. Sensores: cobertura inicial, máscara terrestre e historia de pérdida observada.

La división temporal del piloto en los notebooks 01–03 es histórica. Para el experimento multizona rige `config/hito2.yaml`. Este cuaderno reconstruye el resultado publicado a partir de sus corridas.

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np
from IPython.display import display

raiz = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'config/hito2.yaml').exists())
os.chdir(raiz)
sys.path.insert(0, str(raiz / 'scripts'))
import linea_base_multizona as base

cfg, huella = base.cargar_config('config/hito2.yaml')
zonas = cfg['zonas']
celdas = cfg['celdas']
salida = Path('outputs/ruta_espacial')
cache = Path('data/processed/ruta_espacial')
salida.mkdir(parents=True, exist_ok=True)
cache.mkdir(parents=True, exist_ok=True)

In [2]:
print('SHA-256 del protocolo (prefijo):', huella)
display(pd.read_csv('outputs/hito2/tamizaje_zonas.csv'))
print(cfg['division_temporal'])
print(cfg['umbrales_exito_ruta_completa'])

SHA-256 del protocolo (prefijo): a194e6a72547


,zona,lon_min,lat_min,lon_max,lat_max,alertas_2022,prevalencia_2022_%,alertas_2023,prevalencia_2023_%,alertas_2024,prevalencia_2024_%,celdas_bosque_2024,cumple,rol_candidato,rol_final
0,guaviare,-73.5,1.5,-72.5,2.5,1178,9.56,656,5.33,926,7.52,12316,True,entrenamiento,entrenamiento
1,macarena,-74.5,2.0,-73.5,3.0,2276,18.49,509,4.13,1933,15.70,12311,True,entrenamiento,entrenamiento
2,chaira,-75.0,0.5,-74.0,1.5,2224,18.07,984,8.00,2159,17.55,12290,True,retenida,retenida
3,putumayo,-77.0,0.0,-76.0,1.0,3713,30.14,1427,11.58,1495,12.13,12321,True,retenida,retenida
4,bajo_cauca,-75.5,7.0,-74.5,8.0,1287,10.46,1390,11.30,904,7.35,12298,True,retenida,descartada
5,catatumbo,-73.5,8.0,-72.5,9.0,1446,11.76,1077,8.76,857,6.97,12300,True,retenida,descartada


{'entrenamiento': [2004, 2022], 'validacion': [2023, 2024], 'prueba': [2025]}
{'pr_auc': 'media en retenidas >= 1.10 x línea base, y no menor que la línea base en ninguna zona', 'precision_top20': 'media en retenidas >= 0.75, y mayor que persistencia', 'recall_con_fpr': 'recall >= 0.75 con FPR <= 0.15', 'tiempo_respuesta': '<= 300 s en un recuadro de 1°x1°'}


In [3]:
corridas = pd.read_csv('outputs/hito2/linea_base_corridas.csv')
rf = corridas.query("modelo == 'linea_base_rf'")
esperado = {(z, s) for z in zonas['retenidas'] for s in cfg['linea_base']['semillas']}
assert set(zip(rf.zona, rf.semilla)) == esperado and len(rf) == len(esperado)
metricas = ['pr_auc', 'top_k', 'recall', 'fpr']
display(corridas.groupby(['modelo', 'zona'])[metricas].mean().round(4))
macro = corridas.groupby(['modelo', 'semilla'])[metricas].mean()
display(macro.groupby('modelo').agg(['mean','std']).round(4))
print('AP media mínima para la ruta:', round(1.10 * macro.loc['linea_base_rf'].pr_auc.mean(), 5))

pr_auc   top_k  recall     fpr
modelo        zona                                    
azar          chaira    0.1535  0.1535     NaN     NaN
              putumayo  0.1352  0.1352     NaN     NaN
linea_base_rf chaira    0.5540  0.7500  0.7037  0.1642
              putumayo  0.3192  0.6300  0.8161  0.4232
persistencia  chaira    0.4655  0.7500  0.5188  0.1128
              putumayo  0.2982  0.6500  0.3037  0.0928

pr_auc           top_k          recall             fpr        
                 mean     std    mean     std    mean     std    mean     std
modelo                                                                       
azar           0.1444     NaN  0.1444     NaN     NaN     NaN     NaN     NaN
linea_base_rf  0.4366  0.0006  0.6900  0.0224  0.7599  0.0016  0.2937  0.0007
persistencia   0.3818     NaN  0.7000     NaN  0.4113     NaN  0.1028     NaN

AP media mínima para la ruta: 0.4803


**Decisión de diseño.** En Putumayo el RF marca como positivas unas 42 de cada 100 celdas negativas. En las dos zonas el top-20 medio queda por debajo de persistencia. El candidato incorpora la proximidad a focos y la evolución local para intentar ordenar mejor las visitas. El umbral seguirá fijándose exclusivamente en validación. Con solo 20 visitas, un acierto mueve cinco puntos porcentuales el resultado de una zona.